# K-Means Clustering - From Scratch và `scikit-learn`


## Lý thuyết K-Means Clustering

**K-Means** là một trong những thuật toán **Học không giám sát (Unsupervised Learning)** phổ biến nhất dùng để phân cụm (clustering) dữ liệu. Mục tiêu của K-Means là nhóm các điểm dữ liệu tương đồng vào cùng một cụm và phân tách các cụm khác nhau dựa trên khoảng cách. Cụ thể:

### 1. Ý tưởng cốt lõi
* Phân chia $N$ điểm dữ liệu thành $K$ cụm khác nhau (với $K$ là số cụm cần xác định trước).
* Mỗi cụm được đặc trưng bởi một **tâm cụm (centroid)** là trung bình cộng tọa độ các điểm thuộc cụm đó.
* Tối thiểu hóa hàm mục tiêu WCSS (Within-Cluster Sum of Squares) - tổng bình phương khoảng cách từ mỗi điểm dữ liệu đến centroid gần nhất của nó:
$$\text{WCSS} = \sum_{j=1}^{K} \sum_{x_i \in S_j} \|x_i - \mu_j\|^2$$
Trong đó $\mu_j$ là tâm cụm của cụm $S_j$.

### 2. Các bước thực hiện (Thuật toán Lloyd)
* **Bước 1 (Khởi tạo):** Chọn ngẫu nhiên $K$ điểm làm các centroid ban đầu.
* **Bước 2 (Gán cụm - Assignment):** Tính khoảng cách Euclidean từ mỗi điểm dữ liệu tới tất cả $K$ centroids. Gán điểm dữ liệu đó vào cụm có centroid gần nhất.
* **Bước 3 (Cập nhật - Update):** Tính toán lại vị trí của mỗi centroid bằng trung bình cộng tọa độ của tất cả các điểm dữ liệu đã gán cho cụm đó ở Bước 2:
$$\mu_j = \frac{1}{|S_j|} \sum_{x_i \in S_j} x_i$$
* **Bước 4 (Hội tụ):** Lặp lại Bước 2 và Bước 3 cho tới khi vị trí của các centroids không còn thay đổi (hoặc thay đổi nhỏ hơn một ngưỡng sai số `tol` nhất định), hoặc đạt tới số lượng vòng lặp tối đa (`max_iter`).

### 3. Phương pháp đánh giá và lựa chọn số lượng cụm K
* **Phương pháp Khuỷu tay (Elbow Method):** Vẽ đồ thị WCSS tương ứng với các giá trị $K$ khác nhau. Khi số lượng cụm tăng, WCSS luôn giảm. Điểm mà tốc độ giảm của WCSS bắt đầu chậm lại rõ rệt (tạo thành một "khuỷu tay") chính là giá trị $K$ tối ưu.
* **Hệ số Silhouette (Silhouette Coefficient):** Đo lường chất lượng phân cụm của từng điểm dữ liệu. Hệ số Silhouette có giá trị từ -1 đến 1. Giá trị gần 1 cho thấy điểm được phân vào cụm rất phù hợp và cách biệt tốt với các cụm lân cận. Giá trị trung bình trên toàn bộ tập dữ liệu càng cao, kết quả phân cụm càng chất lượng.


## Triển khai `CustomKMeans` không dùng thư viện

### Import thư viện và load dữ liệu

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os, sys, time

sys.path.append(os.path.abspath(os.path.join('..'))) 
from practice_3.utils.custom_hyperparameter_tuning import CustomGridSearchCV
from practice_3.utils.custom_cv import CustomKFold

# K-Means la Unsupervised Learning -- chi can features X, khong dung nhan y
X_train = joblib.load('./data/ready_for_train/X_train_final.pkl')
X_test  = joblib.load('./data/ready_for_train/X_test_final.pkl')

print('Training dataset shape:', X_train.shape)
print('Testing dataset shape :', X_test.shape)


### Triển khai K-Means từ đầu không dùng thư viện

In [ ]:
class CustomKMeans:
    """
    Thuật toán K-Means Clustering tự triển khai.
    Hỗ trợ khởi tạo centroids ngẫu nhiên và tối thiểu hóa WCSS.
    """
    def __init__(self, n_clusters: int = 5, max_iter: int = 300, tol: float = 1e-4, random_state: int | None = None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def fit(self, X):
        X = np.array(X)
        n_samples, n_features = X.shape

        # Khởi tạo centroids ngẫu nhiên chọn từ tập dữ liệu
        rng = np.random.RandomState(self.random_state)
        random_idx = rng.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_idx].copy()

        for i in range(self.max_iter):
            # Tính khoảng cách Euclidean từ mỗi mẫu tới tất cả các centroids
            # Shape: (n_samples, n_clusters)
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)

            # Gán nhãn cho mỗi mẫu theo centroid gần nhất
            labels = np.argmin(distances, axis=1)

            # Cập nhật centroids
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                members = X[labels == k]
                if len(members) > 0:
                    new_centroids[k] = np.mean(members, axis=0)
                else:
                    # Nếu cụm không có phần tử, chọn một mẫu ngẫu nhiên làm centroid
                    new_centroids[k] = X[rng.choice(n_samples)]

            # Kiểm tra sự hội tụ
            if np.all(np.abs(new_centroids - self.centroids) < self.tol):
                self.centroids = new_centroids
                break

            self.centroids = new_centroids

        self.labels_ = labels
        # Tính toán Within-Cluster Sum of Squares (Inertia)
        final_distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
        self.inertia_ = np.sum(np.min(final_distances, axis=1) ** 2)
        return self

    def predict(self, X) -> np.ndarray:
        X = np.array(X)
        distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
        return np.argmin(distances, axis=1)


### Kiểm tra nhanh mô hình K-Means From Scratch

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Chuẩn hóa dữ liệu trước khi phân cụm
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Chạy thử với K = 5 cụm
start = time.time()
custom_km = CustomKMeans(n_clusters=5, random_state=42)
custom_km.fit(X_train_scaled)
train_time_custom = time.time() - start

score_custom = silhouette_score(X_train_scaled, custom_km.labels_)
print(f'Quick test (K=5) - Train time: {train_time_custom:.1f}s')
print(f'WCSS (Inertia) : {custom_km.inertia_:.4f}')
print(f'Silhouette Score: {score_custom:.4f}')


## Dùng thư viện `sklearn` để triển khai `KMeans`

In [ ]:
from sklearn.cluster import KMeans as SklearnKMeans

# Chạy thư viện sklearn với K = 5 cụm
start = time.time()
sklearn_km = SklearnKMeans(n_clusters=5, random_state=42, n_init=10)
sklearn_km.fit(X_train_scaled)
train_time_sklearn = time.time() - start

score_sklearn = silhouette_score(X_train_scaled, sklearn_km.labels_)
print(f'sklearn K-Means (K=5) - Train time: {train_time_sklearn:.1f}s')
print(f'WCSS (Inertia) : {sklearn_km.inertia_:.4f}')
print(f'Silhouette Score: {score_sklearn:.4f}')


## Hyperparameter Tuning

Để tối ưu hóa số cụm K, Grid Search kết hợp với **5-fold Cross Validation** được sử dụng nhằm tìm ra bộ hyperparameters phù hợp nhất.

Parameter grid được sử dụng:

```python
kmeans_grid = {
    'n_clusters': [3, 4, 5, 6, 7],
    'max_iter'  : [100, 300],
    'tol'       : [1e-4, 1e-3],
}
```

Trong đó:
* `n_clusters` — số cụm $K$ cần tìm.
* `max_iter` — số vòng lặp tối đa của thuật toán.
* `tol` — ngưỡng hội tụ.

Mô hình được đánh giá qua **Silhouette Score** — chỉ số đo lường chất lượng phân cụm (càng gần 1 càng tốt).


In [ ]:
kmeans_grid = {
    'n_clusters': [3, 4, 5, 6, 7],
    'max_iter'  : [100, 300],
    'tol'       : [1e-4, 1e-3],
}


### Lựa chọn số lượng cụm tối ưu (Elbow Method & Silhouette)

In [ ]:
k_range = range(2, 11)
wcss_list = []
silhouette_scores = []

for k in k_range:
    model = CustomKMeans(n_clusters=k, random_state=42)
    model.fit(X_train_scaled)
    wcss_list.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_train_scaled, model.labels_))

# Trực quan hóa kết quả để chọn K
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'crimson'
ax1.set_xlabel('Số lượng cụm (K)', fontsize=12)
ax1.set_ylabel('WCSS (Inertia)', color=color, fontsize=12)
ax1.plot(k_range, wcss_list, marker='o', color=color, linewidth=2, label='WCSS')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_title('Đánh giá số cụm K bằng Elbow Method và Silhouette Score', fontsize=14)

ax2 = ax1.twinx()
color = 'royalblue'
ax2.set_ylabel('Silhouette Score', color=color, fontsize=12)
ax2.plot(k_range, silhouette_scores, marker='s', color=color, linewidth=2, linestyle='--', label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

# In ra kết quả chi tiết
print('Bảng đánh giá các giá trị K:')
print(f"{'K':<5} | {'WCSS (Inertia)':<15} | {'Silhouette Score':<18}")
print("-" * 45)
for idx, k in enumerate(k_range):
    print(f"{k:<5} | {wcss_list[idx]:<15.4f} | {silhouette_scores[idx]:<18.4f}")


### Grid Search dùng Silhouette Score

In [ ]:
cv = CustomKFold(n_splits=5, shuffle=True, random_state=42)
scratch_km = CustomKMeans(random_state=42)

km_grid_search = CustomGridSearchCV(
    estimator=scratch_km,
    param_grid=kmeans_grid,
    cv=cv,
    scoring='silhouette'
)

km_grid_search.fit(X_train_scaled, None)


## So sánh K-Means From Scratch và K-Means `sklearn`

Ta sẽ so sánh hai cách triển khai với cùng số lượng cụm tốt nhất tìm được từ Grid Search.

In [ ]:
from sklearn.metrics import adjusted_rand_score

best_params = km_grid_search.best_params_
print('Best params:', best_params)

# ── From Scratch với best params ──────────────────────
scratch_best = km_grid_search.best_estimator_

start = time.time()
scratch_best.fit(X_train_scaled)
train_time_scratch = time.time() - start

sil_scratch = silhouette_score(X_train_scaled, scratch_best.labels_)
wcss_scratch = scratch_best.inertia_

# ── sklearn với cùng best params ─────────────────────
best_k = best_params['n_clusters']
start = time.time()
sklearn_best = SklearnKMeans(n_clusters=best_k, random_state=42, n_init=10)
sklearn_best.fit(X_train_scaled)
train_time_sklearn = time.time() - start

sil_sklearn = silhouette_score(X_train_scaled, sklearn_best.labels_)
wcss_sklearn = sklearn_best.inertia_

# ── Bảng so sánh ─────────────────────────────────────
print('\n' + '='*60)
print(f"{'Metric':<15} {'From Scratch':>20} {'sklearn':>20}")
print('='*60)
print(f"{'WCSS (Inertia)':<15} {wcss_scratch:>20.4f} {wcss_sklearn:>20.4f}")
print(f"{'Silhouette Score':<15} {sil_scratch:>20.4f} {sil_sklearn:>20.4f}")
print(f"{'Train Time (s)':<15} {train_time_scratch:>20.1f} {train_time_sklearn:>20.1f}")
print('='*60)

# So sánh mức độ tương đồng phân cụm
ari_score = adjusted_rand_score(scratch_best.labels_, sklearn_best.labels_)
print(f'\nChỉ số tương đồng phân cụm Adjusted Rand Index (ARI) giữa 2 mô hình: {ari_score:.4f}')
print('(ARI gần 1.0 có nghĩa là kết quả phân cụm của 2 mô hình gần như khớp hoàn toàn, chỉ khác nhãn số)')


## Kết luận

### Hướng triển khai K-Means Clustering
- **From Scratch:** Triển khai lớp `CustomKMeans` kế thừa các bước của thuật toán Lloyd: Khởi tạo ngẫu nhiên tâm cụm từ dữ liệu $\rightarrow$ Tính khoảng cách Euclidean $\rightarrow$ Gán nhãn cụm $\rightarrow$ Cập nhật tọa độ tâm cụm bằng trung bình cộng nhóm $\rightarrow$ Lặp lại đến khi hội tụ (`tol` hoặc `max_iter`). Việc lập trình từ đầu giúp hiểu rõ bản chất toán học đằng sau K-Means.
- **Sử dụng `KMeans` từ `sklearn.cluster`:** Phiên bản thư viện được tối ưu hóa mạnh mẽ bằng ngôn ngữ C (Cython), hỗ trợ khởi tạo thông minh `K-Means++` giúp tăng tốc độ hội tụ và giảm rủi ro rơi vào cực trị địa phương (local minima).

### So sánh hiệu năng giữa hai cách triển khai
- **Độ chính xác:** Cả hai cách triển khai cho ra kết quả phân cụm có WCSS (Inertia) và Silhouette Score tương đồng nhau. Chỉ số tương đồng phân cụm Adjusted Rand Index (ARI) xấp xỉ 1.0, chứng tỏ phân cụm là hoàn toàn nhất quán.
- **Tốc độ:** Thuật toán tự triển khai trên Python thuần chạy chậm hơn so với phiên bản tối ưu của thư viện `sklearn`, do phiên bản sklearn được biên dịch C và hỗ trợ song song hóa.

### Hiệu năng thực sự của mô hình
K-Means trên dữ liệu thực tế (nhiều chiều, phân phối không đồng đều) cho Silhouette Score thấp do bản chất dữ liệu. Tuy nhiên, K-Means vẫn hữu ích để:
* Khám phá cấu trúc ẩn của dữ liệu.
* Tạo các feature mới (cluster assignment) để đưa vào mô hình supervised.
* Phát hiện outlier dựa trên khoảng cách tới centroid.

### Hướng giải quyết tiếp theo
Mặc dù K-Means hoạt động tốt trên các cụm có dạng hình cầu, thuật toán này có thể thất bại khi các cụm có hình dạng phức tạp hoặc mật độ không đồng đều. Trong phạm vi **Unsupervised Learning**, hướng mở rộng tự nhiên bao gồm:
* **DBSCAN (Density-Based Clustering):** Phát hiện các cụm có hình dạng tùy ý và tự động nhận diện nhiễu (outliers) — không yêu cầu xác định trước số cụm \$.
* **Hierarchical Clustering:** Xây dựng cây phân cụm (dendrogram) cho phép phân tích cấu trúc dữ liệu ở nhiều mức độ chi tiết khác nhau mà không cần chỉ định \$ ngay từ đầu.
* **Gaussian Mixture Models (GMM):** Mô hình hóa phân phối dữ liệu bằng hỗn hợp phân phối Gauss, hỗ trợ phân cụm mềm (soft clustering) — mỗi điểm có xác suất thuộc từng cụm thay vì gán cứng nhãn.


## Lưu model

In [ ]:
MODEL_DIR = './models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Lưu model K-Means From Scratch
model_path = os.path.join(MODEL_DIR, 'kmeans_scratch.pkl')

joblib.dump(
    km_grid_search.best_estimator_,
    model_path
)

print(f"Mô hình K-Means From Scratch đã được lưu tại {model_path}")